# KG pipeline — resolution, canonicalization, fusion, ETO evaluation (Person 2)
Inputs: Person 1's `wiki_claims.jsonl`, your `sec_claims.jsonl` (v3), the company registry. Outputs go to `data/kg/`.
Run top to bottom. LLM calls are cached in `data/kg/cache/`, so re-running is free unless inputs or prompts change.
Calls for a full run: Gemini ~110 (canonicalization) + ~20 (merge). Judge model (Groq, `openai/gpt-oss-120b`, free tier): ~60 (ETO judge) + ~45 (edge check).
Colab secrets needed: `GeminiAPIKey` and `GroqAPIKey` (free key from console.groq.com). Never paste keys into cells.

In [ ]:
!pip install -q openai google-genai sentence-transformers rapidfuzz pydantic pycountry scikit-learn
from google.colab import drive
drive.mount('/content/drive')
import os, sys
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/YOUR_FOLDER/semiconductor_kg")
CODE = ROOT / "kg_pipeline"                      # upload the kg_pipeline folder here
os.environ["KG_ROOT"] = str(ROOT)
sys.path.insert(0, str(CODE)); os.chdir(CODE)
import importlib, kg_common as K
importlib.reload(K)
print(K.ROOT.exists(), K.KG_DIR)

### 1. Inputs
If Person 1's `wiki_claims.jsonl` is somewhere else, put its path in `WIKI` below. ETO's public CSVs are downloaded once into `data/eto/` (evaluation only).

In [ ]:
WIKI = None   # e.g. Path("/content/drive/MyDrive/.../Person 1 submissions/wiki_claims.jsonl")
if WIKI: K.WIKI_CLAIMS_CANDIDATES.insert(0, Path(WIKI))
import urllib.request
K.ETO_DIR.mkdir(parents=True, exist_ok=True)
for f in ["inputs", "providers", "provision", "sequence", "stages"]:
    p = K.ETO_DIR / f"{f}.csv"
    if not p.exists():
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/georgetown-cset/eto-chip-explorer/main/data/{f}.csv", p)
reg, reg_path = K.load_registry()
wiki, sec, bad = K.load_claims()
print("wiki claims:", len(wiki), "| sec claims:", len(sec), "| Contract A violations:", len(bad))
print("registry:", reg_path, len(reg), "companies | wiki file:", K.wiki_claims_path())

### 2. Canonicalize offerings into product families (LLM, never sees ETO)
**Pilot first:** keep `PILOT = True` to run only the four equipment makers (~15 calls), look at the families below, then set `PILOT = False` for the full run. **The pilot only covers 4 companies:** run the full canonicalization before build_kg / the evaluation, otherwise every other company stays at one node per mention.

In [ ]:
import canonicalize_offerings as C
importlib.reload(C)
PILOT = False
C.ONLY_COMPANIES = {"company_008", "company_011", "company_012", "company_014"} if PILOT else set()
resolver = K.CompanyResolver(reg, K.KG_DIR / "company_id_ledger.json")
doc_company, _ = K.sec_doc_company(reg)
rows, fams, stats = C.run(wiki + sec, resolver, doc_company)
resolver.save_ledger()
print(stats)

In [ ]:
# Look at the result: families per company, with the mentions they absorbed
import collections, json
by = collections.defaultdict(list)
for r in rows:
    by[r["company_id"]].append((r["family"], r["mention"], r["grounding"], r["specificity"]))
for cid in sorted(by)[:6]:
    print("\n==", resolver.entities[cid]["canonical_name"])
    fam_groups = collections.defaultdict(list)
    for f, m, g, s in by[cid]:
        fam_groups[f].append(m + ("*" if g == "knowledge" else ""))
    for f, ms in sorted(fam_groups.items(), key=lambda kv: -len(kv[1])):
        print(f"  {f!s:55} <- {ms[:6]}")
print("\n* = family needed outside knowledge (brand/model name)")

### 3. Entity resolution + fusion → files for Person 1's Neo4j loader

In [ ]:
import build_kg as B
importlib.reload(B)
report = B.main()

### 4. ETO evaluation (semantic: LLM judge + embeddings)
Builds the reference from ETO, runs the judge per company (cached), writes `data/kg/eval/`.
Then open `eto_label_sheet.csv`, fill `human_label` (yes / partial / no) for the 60 rows **without looking at the judge column first if you can** (hide it), save as `eto_label_sheet_labeled.csv` in the same folder, and run the last cell.

Labels: **yes** = graph has it (same or more specific; include the original wordings in `kg_all_offerings`), **partial** = only a broader offering, **no** = not there or only related. Groq free tier: ~200k tokens/day, about one full judge pass; re-running the cell resumes from the cache.

In [ ]:
import eto_reference as ER, evaluate_eto as E
importlib.reload(ER); importlib.reload(E)
summary = E.main()

In [ ]:
# after labelling: judge agreement (kappa) + calibrated embedding threshold
summary = E.main(calibrate=True)

### 5. Edge audit: precision by relation / source / entity type, and confidence calibration
ETO only checks (company, input) pairs. This audits every relation type and whether entities were recognised correctly.
1. Run the next cell: it writes `eval/edge_audit_sheet.csv` (80 edges, 10 per relation, split across sources).
2. Label `edge_correct` (yes/no/unsure), `subject_ok`, `object_ok`, `error_type`; save as `edge_audit_sheet_labeled.csv`.
3. The LLM check (judge model) scales the audit to ~600 edges; it is only trusted if it agrees with your labels (kappa).

In [ ]:
import audit_edges as A
importlib.reload(A)
A.make_sheet()          # human sheet (no LLM)

In [ ]:
import shutil, kg_common as K, audit_edges as A
shutil.copy(K.EVAL_DIR/"edge_llm_check.csv", K.EVAL_DIR/"edge_llm_check_20b.csv")   # keep the 20B run
A.llm_check(n=500, model="openai/gpt-oss-120b")
A.score()

In [ ]:
# after labelling edge_audit_sheet_labeled.csv
import importlib, audit_edges; importlib.reload(audit_edges); audit_edges.score()
rep = A.score()
# If the measured per-source precision differs from build_kg.SOURCE_PRIOR, update it there and re-run build_kg.

### 6. Deliver
Copies the graph files and reports to `Person 2 submissions/kg_v1/`.

In [ ]:
import shutil
DEST = ROOT.parent.parent / "Person 2 submissions" / "kg_v1"
DEST.mkdir(parents=True, exist_ok=True)
for f in ["canonical_entities.jsonl", "resolved_claims.jsonl", "fused_edges.jsonl", "documents.jsonl", "dropped_claims.jsonl",
          "offering_canon.jsonl", "offering_families.json", "edge_scores.jsonl", "kg_build_report.md", "kg_build_report.json"]:
    if (K.KG_DIR / f).exists(): shutil.copy(K.KG_DIR / f, DEST / f)
for f in ["eto_eval_report.md", "eto_eval_summary.json", "eto_eval_pairs.csv", "eto_reference_pairs.csv", "eto_company_map.csv",
          "edge_audit_report.md", "edge_audit_report.json", "edge_llm_check.csv"]:
    if (K.EVAL_DIR / f).exists(): shutil.copy(K.EVAL_DIR / f, DEST / f)
print(sorted(p.name for p in DEST.iterdir()))